<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 04 · REAL-TIME ANALYTICS WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">Model Data for Analytical Workloads</div>
  <p style="color:#475569;font-size:15px;line-height:1.7;max-width:900px;margin:0">Turn a source contract and analytical requirements into typed Doris tables with an explicit grain and Table Model.</p>
  <span style="display:inline-block;border:1px solid #99f6e4;border-radius:4px;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:14px;font-size:12px">Doris 4.1.3 · Grain · Data Types · NULL and DEFAULT · Duplicate Key · Aggregate Key</span>
</div>

By the end of this lab, you will have separated permissive source data from a typed internal-table contract and built a compact daily metrics table from the Level 1 event history. Run the cells in order.

### Initialize the Lab

Run the next initialization cell before Section 1. It loads the shared course helper, installs the same output styles used in Level 1, and creates the `lab` object used by every later code cell.

Run it again whenever you restart the Jupyter kernel. It does **not** start Docker, create a table, or change data. Later cells can reconnect to an existing Doris sandbox. Do not continue until the green **Lab tools are ready** message appears.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT);


## 1. Define the event-detail contract

Module 4 builds new models from the Level 1 `events` table. First confirm that the complete source is available and test its intended grain: **one row represents one original customer event**. If every row has a distinct `event_id`, the row count and distinct-event count are equal.

The source contract also states that identifiers are numeric, `event_time` supports time analysis, `revenue` requires exact decimal arithmetic, and `event_type` and `region` are bounded categories. Those requirements—not the largest value in this particular file—determine the durable column types.

In [ ]:
lab.connect(container="doris", host="127.0.0.1", port=9030)
lab.execute("USE doris_course")

lab.sql("""
SELECT
    COUNT(*) AS source_rows,
    COUNT(DISTINCT event_id) AS distinct_event_ids
FROM events
""", title="Source grain check");


**Expected result**

| source_rows | distinct_event_ids |
|---:|---:|
| 10,158,080 | 10,158,080 |

The first value confirms that the complete Level 1 dataset is present. Equality between the two values supports the declared event-detail grain: each source row represents one original event. If the table is missing or the count differs, complete Level 1 Lab 1 before continuing.

### Inspect the selected column contract

The next short metadata query shows how the event-detail contract is represented in Doris. Use the Expected result to connect each type to its business and query requirement.

In [ ]:
lab.sql(
    "SHOW FULL COLUMNS FROM events",
    title="Event-detail column contract",
    columns=["Field", "Type", "Null", "Default"],
);


**Expected result**

The metadata result should show the following contract:

| Field family | Selected contract | Requirement that justifies it |
|---|---|---|
| `event_id`, `user_id`, `product_id` | `BIGINT NOT NULL` | Numeric identifiers follow the upstream contract and retain growth headroom. |
| `event_time` | `DATETIME NOT NULL` | Queries filter, group, and order events by time. |
| `event_type` | `VARCHAR(32) NOT NULL` | A bounded category needs room for controlled future values. |
| `region` | `VARCHAR(16) NOT NULL` | A short, bounded category does not need an unbounded string. |
| `revenue` | `DECIMAL(12,2) NOT NULL` | Currency requires exact fixed-point arithmetic. |

The query displays the selected schema; the table above explains the decision. A data type is not selected by copying the current maximum value exactly. It must also satisfy the source contract and expected analytical operations.

## 2. Expose the risk of storing business values as text

Declaring every incoming field as `VARCHAR` allows values that violate the business contract.

A permissive raw table can store `not-a-number` in a field that the business treats as revenue. The insert therefore succeeds even though the row cannot satisfy a monetary contract. The diagnostic query uses `TRY_CAST` to expose that mismatch without stopping the notebook.

This step creates a controlled raw source. The next step converts its valid values into a typed internal table. Comparing the two results shows the difference between **late validation in every query** and **a durable typed schema at the staging boundary**.

In [ ]:
lab.execute("""
CREATE TABLE IF NOT EXISTS modeling_raw_events (
    event_time_text VARCHAR(32) NOT NULL,
    event_id_text VARCHAR(32) NOT NULL,
    user_id_text VARCHAR(32) NOT NULL,
    event_type_text VARCHAR(32) NOT NULL,
    region_text VARCHAR(16) NULL,
    product_id_text VARCHAR(32) NULL,
    revenue_text VARCHAR(32) NOT NULL
)
DUPLICATE KEY(event_time_text, event_id_text)
DISTRIBUTED BY RANDOM BUCKETS 1
PROPERTIES ("replication_num" = "1")
""")

lab.execute("TRUNCATE TABLE modeling_raw_events")

lab.insert("""
INSERT INTO modeling_raw_events VALUES
    ('2020-03-01 09:00:00', '90001', '7001', 'view',     'region_01', '1005115', '0.00'),
    ('2020-03-01 09:01:00', '90002', '7001', 'cart',     'region_01', '1005115', '0.00'),
    ('2020-03-01 09:02:00', '90003', '7001', 'purchase', 'region_01', '1005115', '29.95'),
    ('2020-03-01 09:03:00', '90004', '7002', 'purchase', 'region_03', '13200021', 'not-a-number'),
    ('2020-03-01 09:04:00', '90005', '7003', 'purchase', NULL,        NULL,       '9.99')
""", title="Write the permissive source sample")

lab.sql("""
SELECT
    event_id_text,
    revenue_text,
    TRY_CAST(revenue_text AS DECIMAL(12,2)) AS typed_revenue,
    CASE
        WHEN TRY_CAST(revenue_text AS DECIMAL(12,2)) IS NULL
        THEN 'invalid DECIMAL'
        ELSE 'valid'
    END AS contract_check
FROM modeling_raw_events
ORDER BY event_id_text
""", title="Source contract violations");


**Expected result**

The write accepts all five rows because `revenue_text` is a `VARCHAR`. Event `90004` is reported as `invalid DECIMAL`, with `typed_revenue = NULL`.

This is the risk of an all-text schema: storage succeeds even when a value cannot satisfy the analytical contract. `TRY_CAST` is used here for diagnosis, not as a reason to keep monetary values as text.

## 3. Enforce a typed internal-table contract

The curated table gives every column its analytical type. Valid values are converted once at the staging boundary instead of repeatedly inside later queries. Because one row still represents one original event, the table uses the Duplicate Key model and does not merge repeated key values.

The final source row has no region. Its insert omits the target `region` column, so Doris applies `DEFAULT "unknown"`. Its missing `product_id` remains SQL `NULL` because the column is optional. The malformed revenue row is excluded after the explicit contract check.

In [ ]:
lab.execute("""
CREATE TABLE IF NOT EXISTS modeling_typed_events (
    event_time DATETIME NOT NULL,
    event_id BIGINT NOT NULL,
    user_id BIGINT NOT NULL,
    event_type VARCHAR(32) NOT NULL,
    region VARCHAR(16) NOT NULL DEFAULT "unknown",
    product_id BIGINT NULL,
    revenue DECIMAL(12,2) NOT NULL DEFAULT "0.00"
)
DUPLICATE KEY(event_time, event_id)
DISTRIBUTED BY RANDOM BUCKETS 1
PROPERTIES ("replication_num" = "1")
""")

lab.execute("TRUNCATE TABLE modeling_typed_events")

lab.insert("""
INSERT INTO modeling_typed_events (
    event_time, event_id, user_id, event_type, region, product_id, revenue
)
SELECT
    CAST(event_time_text AS DATETIME),
    CAST(event_id_text AS BIGINT),
    CAST(user_id_text AS BIGINT),
    event_type_text,
    region_text,
    TRY_CAST(product_id_text AS BIGINT),
    TRY_CAST(revenue_text AS DECIMAL(12,2))
FROM modeling_raw_events
WHERE region_text IS NOT NULL
  AND TRY_CAST(revenue_text AS DECIMAL(12,2)) IS NOT NULL
""", title="Write valid typed rows")

lab.insert("""
INSERT INTO modeling_typed_events (
    event_time, event_id, user_id, event_type, product_id, revenue
)
SELECT
    CAST(event_time_text AS DATETIME),
    CAST(event_id_text AS BIGINT),
    CAST(user_id_text AS BIGINT),
    event_type_text,
    TRY_CAST(product_id_text AS BIGINT),
    TRY_CAST(revenue_text AS DECIMAL(12,2))
FROM modeling_raw_events
WHERE region_text IS NULL
  AND TRY_CAST(revenue_text AS DECIMAL(12,2)) IS NOT NULL
""", title="Apply the region default")

lab.sql(
    "SHOW FULL COLUMNS FROM modeling_typed_events",
    title="Typed column contract",
    columns=["Field", "Type", "Null", "Key", "Default"],
)

lab.sql("""
SELECT
    COUNT(*) AS typed_rows,
    SUM(revenue) AS total_revenue,
    SUM(CASE WHEN region = 'unknown' THEN 1 ELSE 0 END) AS defaulted_regions,
    SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS null_product_ids
FROM modeling_typed_events
""", title="Typed contract validation");


**Expected result**

| typed_rows | total_revenue | defaulted_regions | null_product_ids |
|---:|---:|---:|---:|
| 4 | 39.94 | 1 | 1 |

The schema result shows identifiers as `BIGINT`, time as `DATETIME`, revenue as `DECIMAL(12,2)`, region as required with a default, and product ID as nullable. Later SQL can use `SUM(revenue)` directly without converting text.

## 4. Select a Table Model for a reporting grain

Choose a Table Model from the required behavior when the same key values arrive more than once:

| Business object | One row represents | Appropriate Table Model |
|---|---|---|
| Event history | One original customer event | Duplicate Key model |
| Current entity state | The latest state for one business key | Unique Key model |
| Daily metrics | One date, region, and event-type combination | Aggregate Key model |

The fixed daily report needs a coarser grain: one row for each `(event_date, region, event_type)` combination.

`daily_event_metrics` uses those dimensions as its Aggregate Key. The additive measures declare `SUM`, so Doris can combine repeated keys during ingestion, Compaction, and final query aggregation.

Do not define `user_count` as an ordinary `SUM` value: distinct users are not additive because one user can appear in multiple input batches. Use a Bitmap or HLL state when a model requires a mergeable distinct count.

The rollup is tiny and has no independent date-lifecycle requirement, so it uses one Bucket and no explicit Partition. More physical objects would add metadata without helping this workload.

In [ ]:
lab.execute("""
CREATE TABLE IF NOT EXISTS daily_event_metrics (
    event_date DATE NOT NULL,
    region VARCHAR(16) NOT NULL,
    event_type VARCHAR(32) NOT NULL,
    event_count BIGINT SUM NOT NULL DEFAULT "0",
    total_revenue DECIMAL(18,2) SUM NOT NULL DEFAULT "0.00"
)
AGGREGATE KEY(event_date, region, event_type)
DISTRIBUTED BY HASH(region) BUCKETS 1
PROPERTIES ("replication_num" = "1")
""")

lab.execute("TRUNCATE TABLE daily_event_metrics")

lab.insert("""
INSERT INTO daily_event_metrics (
    event_date, region, event_type, event_count, total_revenue
)
SELECT
    TO_DATE(event_time),
    region,
    event_type,
    COUNT(*),
    SUM(revenue)
FROM events
GROUP BY TO_DATE(event_time), region, event_type
""", title="Build the daily reporting grain")

lab.sql("""
SELECT
    COUNT(*) AS metric_rows,
    SUM(event_count) AS represented_event_rows,
    SUM(total_revenue) AS represented_revenue
FROM daily_event_metrics
""", title="Daily model validation");


**Expected result**

The insert reports the time required to aggregate the complete event history.

| Check | Expected value |
|---|---:|
| `metric_rows` | Far fewer than 10,158,080 |
| `represented_event_rows` | 10,158,080 |
| `represented_revenue` | 39,984,455.64 |

The smaller physical row count is correct because the table has a coarser grain. The additive measures reconcile to the detail table.

## 5. Verify equivalent answers at different grains

Both branches answer the same business question for 1 March 2020. The first scans original events and aggregates at query time. The second reads measures already modeled at daily grain.

A summary table is safe only for questions compatible with its dimensions and additive measures. It cannot return individual events or a dimension omitted from its grain.

In [ ]:
lab.sql("""
SELECT
    'detail events' AS source,
    COUNT(*) AS event_count,
    SUM(revenue) AS total_revenue
FROM events
WHERE event_time >= '2020-03-01 00:00:00'
  AND event_time <  '2020-03-02 00:00:00'

UNION ALL

SELECT
    'daily metrics',
    SUM(event_count),
    SUM(total_revenue)
FROM daily_event_metrics
WHERE event_date = '2020-03-01'
ORDER BY source
""", title="Equivalent daily result");


**Expected result**

| source | event_count | total_revenue |
|---|---:|---:|
| daily metrics | 75,259 | 5,670,241.29 |
| detail events | 75,259 | 5,670,241.29 |

The tables are not interchangeable for every query. `events` preserves event-level flexibility; `daily_event_metrics` trades detail for a compact representation of its declared reporting grain.

## 6. Inspect the completed model

The column result exposes Key and Value semantics; the compact design view confirms the selected Table Model and distribution.

In [ ]:
lab.sql(
    "SHOW FULL COLUMNS FROM daily_event_metrics",
    title="Daily metric column contract",
    columns=["Field", "Type", "Null", "Key", "Default", "Extra"],
)

lab.sql(
    "SHOW CREATE TABLE daily_event_metrics",
    title="Daily metric physical design",
    metadata_view="design",
);


**Expected result**

- `event_date`, `region`, and `event_type` are Key columns and define the grain.
- `event_count` and `total_revenue` are Value columns with `SUM` aggregation.
- The compact design result reports the Aggregate Key model, `HASH(region)` distribution, one Bucket, and one replica.
- There is no explicit Partition because this small rollup has no Partition-level lifecycle requirement.

### Stop the Doris sandbox

Run this optional cell to release CPU and memory. It stops Doris but preserves the container, image, named volumes, and Lab 4 tables.

In [ ]:
lab.shell(r"""
set -euo pipefail

docker stop doris
docker inspect --format 'container={{.State.Status}}' doris
""", title="Stop the Doris sandbox");


**Expected result:** Docker reports `container=exited`. The Level 1 baseline and Lab 4 tables remain in the named volumes.

### Restart the Doris sandbox

Run this cell to continue with the same environment. It starts the existing container, reconnects to FE, and verifies the persisted Lab 4 results.

In [ ]:
lab.shell(r"""
set -euo pipefail

docker start doris
docker inspect --format 'container={{.State.Status}} health={{.State.Health.Status}}' doris
""", title="Start the Doris sandbox")

lab.connect(container="doris", host="127.0.0.1", port=9030)
lab.execute("USE doris_course")

lab.sql("""
SELECT
    (SELECT COUNT(*) FROM modeling_typed_events) AS typed_sample_rows,
    (SELECT SUM(event_count) FROM daily_event_metrics) AS represented_event_rows,
    (SELECT SUM(total_revenue) FROM daily_event_metrics) AS represented_revenue
""", title="Recovered Lab 4 models");


**Expected result:** the container returns to `healthy`. The result reports 4 typed sample rows, 10,158,080 represented event rows, and represented revenue of 39,984,455.64.

## Lab complete

You used observed values to justify a schema, detected an invalid value in a permissive text staging table, applied typed columns with explicit NULL and DEFAULT semantics, and selected an Aggregate Key model for a declared daily reporting grain. Detail and summary models produced the same compatible business result while preserving different levels of detail.

Official references: [Table Model Overview](https://doris.apache.org/docs/4.x/table-design/data-model/intro/) · [Table Model Best Practices](https://doris.apache.org/docs/4.x/table-design/data-model/tips/) · [CREATE TABLE](https://doris.apache.org/docs/4.x/sql-manual/sql-statements/table-and-view/table/CREATE-TABLE/) · [CAST and TRY_CAST](https://doris.apache.org/docs/4.x/sql-manual/basic-element/sql-data-types/conversion/cast-expr/) · [Aggregate Key model](https://doris.apache.org/docs/4.x/table-design/data-model/aggregate/)